In [ ]:
import time
import requests
import pandas as pd
from datetime import datetime, timezone
from typing import Optional, Set

API_URL = "https://api.hyperliquid.xyz/info"

# -------------------------------------------------------------------------
# LISTA TICKER (stessa lista del funding)
# -------------------------------------------------------------------------
tickers = [
    'ATOM','REQ','CRV','MAVIA','SAGA','NEAR','MORPHO','MANTA','MOVE','XAI',
    'ETC','DOGE','SOPH','CELO','MAV','POPCAT','SCR','COMP','GMT','SOL','IMX',
    'JUP','RUNE','LAUNCHCOIN','UMA','TRB','USTC','AIXBT','IOTA','VIRTUAL',
    'ALGO','GMX','ANIME','BCH','BIO','BSV','NXPC','MOODENG','TNSR','HBAR',
    'SNX','ZEREBRO','HYPER','SAND','BERA','PURR','GAS','LDO','ONDO','DYDX',
    'FTT','TON','EIGEN','LTC','BLAST','AI16Z','OMNI','AAVE','OGN','SUI',
    'MEME','FXS','NEIROETH','NIL','CFX','ME','XRP','TIA','BNB','NOT','IP',
    'OM','TAO','OP','CAKE','AVAX','kPEPE','GALA','MNT','BOME','SUPER','SEI',
    'VINE','KAS','BABY','STX','S','FARTCOIN','STG','RENDER','ENA','LINK',
    'ARB','ARK','BIGTIME','BTC','ETH','RSR','kDOGS','BRETT','BANANA','XLM',
    'INJ','ENS','AR','DOT','SPX','ETHFI','PAXG','kLUNC','GOAT','kSHIB','FIL',
    'MEW','STRK','TRX','ZK','KAITO','PENGU','kBONK','VVV','ORDI','INIT','APT',
    'REZ','LAYER','ZEN','SUSHI','kFLOKI','ADA','kNEIRO','PEOPLE','ZORA',
    'PENDLE','APE','HYPE','FET','CHILLGUY','MELANIA','GRIFFAIN','PNUT','DOOD',
    'WIF','ACE','ZETA','TRUMP','NEO','JTO','YGG','ZRO','PROMPT','WLD','W',
    'MERL','BLUR','UNI','DYM','MINA','MKR','POLYX','POL','IO','TURBO','PYTH',
    'USUAL','GRASS','ALT','HMSTR','WCT','SYRUP','RESOLV','PROVE','YZY','WLFI',
    'TST','PUMP','LINEA','SKY','ASTER','0G','STBL','AVNT','XPL','ZEC','ICP'
]


# -------------------------------------------------------------------------
# FUNZIONI DI SUPPORTO (copiate dal codice funding)
# -------------------------------------------------------------------------

def safe_post(payload: dict,
              max_retries: int = 5,
              base_delay: float = 1.0) -> Optional[requests.Response]:
    """
    POST verso API_URL gestendo rate-limit (429) con retry e backoff esponenziale.
    """
    attempt = 0
    delay = base_delay

    while True:
        try:
            r = requests.post(API_URL, json=payload, timeout=10)
            # Gestione rate limit
            if r.status_code == 429:
                attempt += 1
                if attempt > max_retries:
                    print(f"Rate limit esaurito per payload={payload.get('type')} {payload.get('coin')}, rinuncio.")
                    return None
                print(f"Rate limited (429) per {payload.get('type')} {payload.get('coin')}, retry fra {delay:.1f}s...")
                time.sleep(delay)
                delay *= 2
                continue

            r.raise_for_status()
            return r

        except Exception as e:
            attempt += 1
            if attempt > max_retries:
                print(f"HTTP error definitivo per {payload.get('type')} {payload.get('coin')}: {e}")
                return None
            print(f"HTTP error per {payload.get('type')} {payload.get('coin')}: {e}, retry fra {delay:.1f}s...")
            time.sleep(delay)
            delay *= 2


def get_perp_universe() -> Optional[Set[str]]:
    """
    Restituisce l'insieme dei nomi dei perps (campo 'name' in 'universe')
    usando type: 'meta'. Se fallisce, restituisce None.
    """
    payload = {"type": "meta"}
    r = safe_post(payload)
    if r is None:
        print("Impossibile ottenere la meta; uso la lista ticker così com'è.")
        return None

    try:
        meta = r.json()
    except Exception as e:
        print(f"Errore parsing meta JSON: {e} | raw: {r.text[:200]}")
        return None

    universe = meta.get("universe", [])
    names = {c.get("name") for c in universe if isinstance(c, dict) and "name" in c}
    return names


def detect_listing_time_ms(coin: str, end_ms: int) -> Optional[int]:
    """
    Stima la 'listing date' del perp come il timestamp del primo record
    disponibile in fundingHistory (più vecchio).
    """
    payload = {
        "type": "fundingHistory",
        "coin": coin,
        "startTime": 0,
        "endTime": end_ms,
    }

    r = safe_post(payload)
    if r is None:
        print(f"Impossibile detectare listing per {coin} (safe_post fallita).")
        return None

    try:
        data = r.json()
    except Exception as e:
        print(f"JSON decode error per listing {coin}: {e} | raw: {r.text[:200]}")
        return None

    if not isinstance(data, list) or len(data) == 0:
        # Nessun funding registrato per questo coin
        return None

    first = data[0]
    t = first.get("time")
    if t is None:
        return None

    return int(t)


def fetch_candles(coin: str,
                  start_ms: int,
                  end_ms: int,
                  interval: str = "1h") -> pd.DataFrame:
    """
    Scarica le candele per `coin` tra start_ms ed end_ms con intervallo `interval`.
    Usa 'candleSnapshot' (non paginato, come nel tuo codice originale).
    """
    payload = {
        "type": "candleSnapshot",
        "req": {
            "coin": coin,
            "interval": interval,
            "startTime": start_ms,
            "endTime": end_ms
        }
    }

    r = safe_post(payload)
    if r is None:
        return pd.DataFrame()

    try:
        data = r.json()
    except Exception as e:
        print(f"JSON decode error per candles {coin}: {e} | raw: {r.text[:200]}")
        return pd.DataFrame()

    if not isinstance(data, list) or len(data) == 0:
        return pd.DataFrame()

    df = pd.DataFrame(data)

    # Deve avere almeno le colonne 't' (time) e 'c' (close)
    if "t" not in df.columns or "c" not in df.columns:
        print(f"Colonne inattese per candles {coin}: {df.columns.tolist()}")
        return pd.DataFrame()

    df["time"] = pd.to_datetime(df["t"], unit="ms", utc=True)
    return df


# -------------------------------------------------------------------------
# MAIN: scarica prezzi giornalieri alle 19:00
# -------------------------------------------------------------------------

def main():
    # Start "globale" minimo (come nel funding)
    global_start = datetime(2025, 6, 5, tzinfo=timezone.utc)
    now = datetime.now(timezone.utc)

    global_start_ms = int(global_start.timestamp() * 1000)
    end_ms = int(now.timestamp() * 1000)

    # 1) Universo dei perps da HL
    perp_universe = get_perp_universe()
    if perp_universe is not None:
        valid_tickers = [t for t in tickers if t in perp_universe]
        missing = sorted(set(tickers) - perp_universe)
        if missing:
            print("Ticker non trovati in universe (probabilmente non perps o nomi diversi):")
            print(", ".join(missing))
    else:
        valid_tickers = tickers

    price_rows = []

    for ticker in valid_tickers:
        print(f"\n=== {ticker} ===")

        # 2) Detect listing time per questo perp (come nel funding)
        listing_ms = detect_listing_time_ms(ticker, end_ms)
        if listing_ms is None:
            print(f"⚠️ Nessun funding (o impossibile trovare listing) per {ticker} — skipped prezzi")
            time.sleep(0.2)
            continue

        # Start effettivo: max(start globale, listing)
        start_ms = max(global_start_ms, listing_ms)

        print(
            f"Listing {ticker}: {datetime.fromtimestamp(listing_ms/1000, tz=timezone.utc)} "
            f" | start effettivo_candles: {datetime.fromtimestamp(start_ms/1000, tz=timezone.utc)}"
        )

        # 3) Scarica le candele 1h dall'effettivo start_ms a end_ms
        df_c = fetch_candles(ticker, start_ms, end_ms, interval="1h")

        if df_c.empty:
            print(f"⚠️ Nessuna candela in range per {ticker} — skipped")
            time.sleep(0.2)
            continue

        # 4) Filtra solo le candele delle 19:00 (ora UTC)
        df_19 = df_c[df_c["time"].dt.hour == 19]

        if df_19.empty:
            print(f"⚠️ Nessuna candela alle 19:00 per {ticker} — skipped (ma candele esistono)")
            time.sleep(0.2)
            continue

        for _, row in df_19.iterrows():
            price_rows.append({
                "perp": ticker,
                "time": row["time"],
                "close": row["c"]
            })

        # Piccolo sleep per non abusare
        time.sleep(0.2)

    if not price_rows:
        print("Nessun dato prezzi scaricato.")
        return

    final_df = pd.DataFrame(price_rows)
    final_df = final_df.sort_values(["perp", "time"])

    out_name = "perps_prices_19.csv"
    final_df.to_csv(out_name, index=False)
    print(f"\n✔️ Saved → {out_name}")


if __name__ == "__main__":
    main()
